In [ ]:
# ============================================
# DB 연결 및 설정
# ============================================

import sqlite3
import pandas as pd

# DB 파일 연결 (없으면 생성됨)
conn = sqlite3.connect("restaurant.db")
cursor = conn.cursor()

# SQLite는 기본적으로 FOREIGN KEY가 꺼져있음 → 반드시 활성화
conn.execute("PRAGMA foreign_keys = ON;")

print("✅ DB 연결 완료")

✅ DB 연결 완료


In [ ]:
# ============================================
# 기존 테이블 삭제 (초기화용)
# 필요할 때만 실행
# ============================================

cursor.executescript("""
DROP TABLE IF EXISTS rel_review_tag;
DROP TABLE IF EXISTS rel_restaurant_tag;
DROP TABLE IF EXISTS rel_restaurant_category;
DROP TABLE IF EXISTS review;
DROP TABLE IF EXISTS tag;
DROP TABLE IF EXISTS category;
DROP TABLE IF EXISTS menu;
DROP TABLE IF EXISTS food;
DROP TABLE IF EXISTS restaurant;
DROP TABLE IF EXISTS user;
""")

conn.commit()

print("🧹 기존 테이블 삭제 완료")

🧹 기존 테이블 삭제 완료


In [ ]:
# ============================================
# USER 테이블 생성
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS user (
    user_code TEXT PRIMARY KEY,
    nickname TEXT,
    created_at TEXT,
    avg_score REAL,
    review_cnt INTEGER,
    follower_cnt INTEGER
);
""")

conn.commit()
print("✅ USER 테이블 생성 완료")

✅ USER 테이블 생성 완료


In [ ]:
# ============================================
# RESTAURANT 테이블 생성
# 위치 기반 검색을 위해 lat/lng 포함
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS restaurant (
    restaurant_code TEXT PRIMARY KEY,
    name TEXT,
    img_link TEXT,
    region TEXT,
    address TEXT,
    lat REAL,
    lng REAL,
    open_time TEXT,
    close_time TEXT,
    tel_no TEXT
);
""")

conn.commit()
print("✅ RESTAURANT 테이블 생성 완료")

✅ RESTAURANT 테이블 생성 완료


In [ ]:
# ============================================
# MENU 테이블 생성
# food_code 제거 → 메뉴는 식당 종속 구조
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS menu (
    menu_code TEXT PRIMARY KEY,
    restaurant_code TEXT,
    name TEXT,
    price INTEGER,
    description TEXT,
    embedding TEXT,
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code)
);
""")

conn.commit()
print("✅ MENU 테이블 생성 완료")

✅ MENU 테이블 생성 완료


In [ ]:
# ============================================
# FOOD 테이블 (선택 유지)
# 음식 카테고리/유사도 분석용
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS food (
    food_code TEXT PRIMARY KEY,
    name TEXT,
    embedding TEXT
);
""")

conn.commit()
print("✅ FOOD 테이블 생성 완료")

✅ FOOD 테이블 생성 완료


In [ ]:
# ============================================
# CATEGORY 테이블
# 고정 분류 (한식, 중식 등)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS category (
    category_code TEXT PRIMARY KEY,
    name TEXT,
    embedding TEXT
);
""")

conn.commit()
print("✅ CATEGORY 테이블 생성 완료")

✅ CATEGORY 테이블 생성 완료


In [ ]:
# ============================================
# TAG 테이블
# 유동적 분류 (데이트, 혼밥, 가성비 등)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS tag (
    tag_code TEXT PRIMARY KEY,
    name TEXT,
    embedding TEXT
);
""")

conn.commit()
print("✅ TAG 테이블 생성 완료")

✅ TAG 테이블 생성 완료


In [ ]:
# ============================================
# RESTAURANT - CATEGORY (다대다 관계)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS rel_restaurant_category (
    restaurant_code TEXT,
    category_code TEXT,
    PRIMARY KEY (restaurant_code, category_code),
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code),
    FOREIGN KEY (category_code) REFERENCES category(category_code)
);
""")

conn.commit()
print("✅ RESTAURANT-CATEGORY 관계 생성 완료")

✅ RESTAURANT-CATEGORY 관계 생성 완료


In [ ]:
# ============================================
# RESTAURANT - TAG (다대다 관계)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS rel_restaurant_tag (
    restaurant_code TEXT,
    tag_code TEXT,
    PRIMARY KEY (restaurant_code, tag_code),
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code),
    FOREIGN KEY (tag_code) REFERENCES tag(tag_code)
);
""")

conn.commit()
print("✅ RESTAURANT-TAG 관계 생성 완료")

✅ RESTAURANT-TAG 관계 생성 완료


In [ ]:
# ============================================
# REVIEW 테이블
# 평점 + 세부 평가 + 텍스트 + embedding 포함
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS review (
    review_code TEXT PRIMARY KEY,
    restaurant_code TEXT,
    user_code TEXT,
    score REAL,
    taste_level INTEGER,
    price_level INTEGER,
    service_level INTEGER,
    content TEXT,
    embedding TEXT,
    created_at TEXT,
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code),
    FOREIGN KEY (user_code) REFERENCES user(user_code)
);
""")

conn.commit()
print("✅ REVIEW 테이블 생성 완료")

✅ REVIEW 테이블 생성 완료


In [ ]:
# ============================================
# REVIEW - TAG (다대다 관계)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS rel_review_tag (
    review_code TEXT,
    tag_code TEXT,
    PRIMARY KEY (review_code, tag_code),
    FOREIGN KEY (review_code) REFERENCES review(review_code),
    FOREIGN KEY (tag_code) REFERENCES tag(tag_code)
);
""")

conn.commit()
print("✅ REVIEW-TAG 관계 생성 완료")

✅ REVIEW-TAG 관계 생성 완료


In [ ]:
# ============================================
# USER - RESTAURANT LIKE (찜 기능)
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS user_restaurant_like (
    user_code TEXT,
    restaurant_code TEXT,
    PRIMARY KEY (user_code, restaurant_code),
    FOREIGN KEY (user_code) REFERENCES user(user_code),
    FOREIGN KEY (restaurant_code) REFERENCES restaurant(restaurant_code)
);
""")

conn.commit()
print("✅ USER LIKE 테이블 생성 완료")

✅ USER LIKE 테이블 생성 완료


In [ ]:
# ============================================
# REVIEW IMAGE 테이블
# ============================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS review_image (
    image_id INTEGER PRIMARY KEY AUTOINCREMENT,
    review_code TEXT,
    image_url TEXT,
    FOREIGN KEY (review_code) REFERENCES review(review_code)
);
""")

conn.commit()
print("✅ REVIEW IMAGE 테이블 생성 완료")

✅ REVIEW IMAGE 테이블 생성 완료


In [ ]:
# ============================================
# 생성된 테이블 확인
# ============================================

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,user
1,restaurant
2,menu
3,food
4,category
5,tag
6,rel_restaurant_category
7,rel_restaurant_tag
8,review
9,rel_review_tag
